<a href="https://colab.research.google.com/github/Vinit-4689/Flood-Risk-Mapping-using-GIS/blob/main/notebook/Flood_Risk_Mapping_using_GIS%20(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project FRM-GIS

## 1. Initialize Google Earth Engine

Load the Google Cloud Project ID from Colab Secrets, authenticate
with Google Earth Engine, and verify that the connection works.

In [1]:
%pip install -q geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 36.9 MB/s eta 0:00:00


In [2]:
!pip install earthengine-api geemap

In [3]:

import ee
from google.colab import userdata

PROJECT_ID = userdata.get("PROJECT_ID")

if not PROJECT_ID:
    raise ValueError(
        "PROJECT_ID not found. Check Colab Secrets and notebook access."
    )

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print(ee.String("Earth Engine connection successful").getInfo())


Earth Engine connection successful


In [4]:
# Verify that the SRTM 30 m elevation dataset is accessible
srtm = ee.Image("USGS/SRTMGL1_003")

print("Dataset:", "SRTM 30 m DEM")
print("Available bands:", srtm.bandNames().getInfo())
print("Dataset access verified.")

Dataset: SRTM 30 m DEM
Available bands: ['elevation']
Dataset access verified.


##2.1 Inspect candidate watershed boundaries

Use HydroBASINS Level 7 to identify candidate catchments around
the Pamba River Basin. Inspect the results before selecting an AOI.

In [5]:
import geemap

# Approximate search window around the Pamba Basin.
# This is only for finding candidate catchments, NOT the final AOI.
search_area = ee.Geometry.Rectangle([76.2, 8.5, 77.4, 10.2])

# Load candidate watershed polygons
basins = ee.FeatureCollection(
    "WWF/HydroSHEDS/v1/Basins/hybas_7"
).filterBounds(search_area)

print("Candidate basin polygons:", basins.size().getInfo())

# Inspect available attributes
print(
    "First candidate attributes:",
    basins.first().toDictionary().getInfo()
)

Candidate basin polygons: 14
First candidate attributes: {'COAST': 0, 'DIST_MAIN': 363.1, 'DIST_SINK': 363.1, 'ENDO': 0, 'HYBAS_ID': 4071150410, 'MAIN_BAS': 4070028760, 'NEXT_DOWN': 4071149830, 'NEXT_SINK': 4070028760, 'ORDER': 2, 'PFAF_ID': 4538029, 'SORT': 5322, 'SUB_AREA': 1543, 'UP_AREA': 1543.4}


In [6]:
Map = geemap.Map(
    basemap="Esri.WorldTopoMap"
)

Map.centerObject(search_area, 8)

Map.addLayer(
    search_area,
    {"color": "yellow"},
    "Search window"
)

Map.addLayer(
    basins.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "HydroBASINS candidates"
)

Map

Map(center=[9.349811297242743, 76.79999999999993], controls=(WidgetControl(options=['position', 'transparent_b…

## 2.2 Inspect HydroBASINS Candidate Attributes

Check the number of candidate features and their available
properties before selecting the Pamba River Basin boundary.

In [7]:
# Count candidate basin features
print("Number of basin candidates:", basins.size().getInfo())

# Inspect available attributes
first_basin = ee.Feature(basins.first())

print("\nAvailable properties:")
print(first_basin.propertyNames().getInfo())

# Display a small sample of attributes without geometries
sample = (
    basins
    .limit(5)
    .select(first_basin.propertyNames(), None, False)
)

print("\nFirst five candidate records:")
print(sample.getInfo()["features"])

Number of basin candidates: 14

Available properties:
['SUB_AREA', 'COAST', 'PFAF_ID', 'DIST_MAIN', 'HYBAS_ID', 'DIST_SINK', 'NEXT_DOWN', 'ORDER', 'ENDO', 'MAIN_BAS', 'NEXT_SINK', 'SORT', 'system:index', 'UP_AREA']

First five candidate records:
[{'type': 'Feature', 'geometry': None, 'id': '0001000000000000052a', 'properties': {'COAST': 0, 'DIST_MAIN': 363.1, 'DIST_SINK': 363.1, 'ENDO': 0, 'HYBAS_ID': 4071150410, 'MAIN_BAS': 4070028760, 'NEXT_DOWN': 4071149830, 'NEXT_SINK': 4070028760, 'ORDER': 2, 'PFAF_ID': 4538029, 'SORT': 5322, 'SUB_AREA': 1543, 'UP_AREA': 1543.4}}, {'type': 'Feature', 'geometry': None, 'id': '0001000000000000070e', 'properties': {'COAST': 0, 'DIST_MAIN': 0, 'DIST_SINK': 0, 'ENDO': 0, 'HYBAS_ID': 4070029250, 'MAIN_BAS': 4070029250, 'NEXT_DOWN': 0, 'NEXT_SINK': 4070029250, 'ORDER': 1, 'PFAF_ID': 4539020, 'SORT': 5351, 'SUB_AREA': 6713.5, 'UP_AREA': 6714.3}}, {'type': 'Feature', 'geometry': None, 'id': '00010000000000000710', 'properties': {'COAST': 0, 'DIST_MAIN': 0,

## 2.3 Compare HydroBASINS Candidates

Create a compact attribute table for all candidate polygons
to support evidence-based selection of the Pamba River Basin.

In [8]:
import pandas as pd

fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "MAIN_BAS",
    "NEXT_DOWN",
    "DIST_MAIN",
    "ORDER"
]

candidate_features = (
    basins
    .select(fields, None, False)
    .getInfo()["features"]
)

candidate_df = pd.DataFrame(
    [feature["properties"] for feature in candidate_features]
)

candidate_df = candidate_df.sort_values(
    by="SUB_AREA",
    ascending=False
).reset_index(drop=True)

display(candidate_df.round(2))

,DIST_MAIN,HYBAS_ID,MAIN_BAS,NEXT_DOWN,ORDER,PFAF_ID,SUB_AREA,UP_AREA
0,0.0,4070029250,4070029250,0,1,4539020,6713.5,6714.3
1,0.0,4070029810,4070029810,0,1,4540200,6376.7,6376.9
2,0.0,4070029440,4070029440,0,1,4539080,5625.5,5625.7
3,0.0,4070029360,4070029360,0,1,4539060,5159.1,5159.1
4,0.0,4070029530,4070029530,0,0,4540101,4728.5,4728.5
5,0.0,4070029720,4070029720,0,1,4540104,3955.6,3955.6
6,0.0,4070029770,4070029770,0,1,4540108,1809.4,1809.5
7,363.1,4071150410,4070028760,4071149830,2,4538029,1543.0,1543.4
8,0.0,4070029680,4070029680,0,1,4540102,1507.1,1507.1
9,0.0,4070029750,4070029750,0,1,4540106,1318.6,1318.9


## 2.6 Geographic Verification of HydroBASINS Candidates

Calculate each candidate's centroid and geometric area to help
identify which polygons overlap the intended Pamba catchment.

In [9]:
def add_geometry_metrics(feature):
    centroid = feature.geometry().centroid(maxError=100)
    coords = centroid.coordinates()

    return feature.set({
        "centroid_lon": coords.get(0),
        "centroid_lat": coords.get(1),
        "geometry_area_km2": feature.geometry().area(maxError=100).divide(1e6)
    })

basins_with_metrics = basins.map(add_geometry_metrics)

metric_fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "centroid_lon",
    "centroid_lat",
    "geometry_area_km2"
]

features = (
    basins_with_metrics
    .select(metric_fields, None, False)
    .getInfo()["features"]
)

geo_df = pd.DataFrame([
    feature["properties"] for feature in features
])

geo_df = geo_df.sort_values(
    "centroid_lat",
    ascending=False
).reset_index(drop=True)

display(geo_df.round(4))

,HYBAS_ID,PFAF_ID,SUB_AREA,UP_AREA,centroid_lat,centroid_lon,geometry_area_km2
0,4071150410,4538029,1543.0,1543.4,10.3886,77.2861,1542.9605
1,4070029810,4540200,6376.7,6376.9,10.0549,76.8511,6376.6909
2,4070029780,4540109,660.9,660.9,9.9732,76.3438,660.9421
3,4070029770,4540108,1809.4,1809.5,9.8943,76.6432,1809.3996
4,4070029250,4539020,6713.5,6714.3,9.8903,77.8653,6713.5544
5,4070029730,4540105,543.9,543.9,9.6691,76.3299,543.8686
6,4070029750,4540106,1318.6,1318.9,9.6245,76.6417,1318.6210
7,4070029760,4540107,87.5,87.5,9.5837,76.4453,87.4622
8,4070029360,4539060,5159.1,5159.1,9.3972,77.7572,5159.1107
9,4070029720,4540104,3955.6,3955.6,9.3345,76.8777,3955.6393


### 2.7 Screen HydroBASINS Candidates by Geographic Extent

Compare each candidate's bounding coordinates with the official
Pamba basin map to identify polygons worth inspecting visually.

This is a preliminary screening step, not final boundary validation.

In [10]:
def add_bbox_metrics(feature):
    ring = ee.List(
        feature.geometry()
        .bounds(maxError=100)
        .coordinates()
        .get(0)
    )

    lons = ring.map(lambda point: ee.List(point).get(0))
    lats = ring.map(lambda point: ee.List(point).get(1))

    return feature.set({
        "min_lon": ee.Number(lons.reduce(ee.Reducer.min())),
        "max_lon": ee.Number(lons.reduce(ee.Reducer.max())),
        "min_lat": ee.Number(lats.reduce(ee.Reducer.min())),
        "max_lat": ee.Number(lats.reduce(ee.Reducer.max()))
    })

basins_bbox = basins.map(add_bbox_metrics)

bbox_fields = [
    "HYBAS_ID", "PFAF_ID", "SUB_AREA",
    "min_lon", "max_lon", "min_lat", "max_lat"
]

bbox_features = (
    basins_bbox
    .select(bbox_fields, None, False)
    .getInfo()["features"]
)

bbox_df = pd.DataFrame([
    f["properties"] for f in bbox_features
])

display(
    bbox_df.sort_values("SUB_AREA")
    .reset_index(drop=True)
    .round(4)
)

,HYBAS_ID,PFAF_ID,SUB_AREA,max_lat,max_lon,min_lat,min_lon
0,4070029760,4540107,87.5,9.6375,76.5083,9.5208,76.3786
1,4070029690,4540103,502.9,9.3125,76.7875,8.9375,76.3792
2,4070029730,4540105,543.9,9.9750,76.4208,9.3167,76.2328
3,4070029780,4540109,660.9,10.1917,76.4795,9.6417,76.1625
4,4070029750,4540106,1318.6,9.8458,76.9208,9.3708,76.3828
5,4070029680,4540102,1507.1,9.1750,77.2625,8.7250,76.5333
6,4071150410,4538029,1543.0,10.7000,77.5083,10.1244,77.0619
7,4070029770,4540108,1809.4,10.1337,76.9417,9.6161,76.3994
8,4070029720,4540104,3955.6,9.6708,77.2833,9.0208,76.3625
9,4070029530,4540101,4728.5,9.0083,77.6500,8.0744,76.5333


## 2.7 Screen HydroBASINS Candidates by Geographic Extent

Compare each candidate's bounding coordinates with the official
Pamba basin map to identify polygons worth inspecting visually.

This is a preliminary screening step, not final boundary validation.

In [11]:
def add_bbox_metrics(feature):
    ring = ee.List(
        feature.geometry()
        .bounds(maxError=100)
        .coordinates()
        .get(0)
    )

    lons = ring.map(lambda point: ee.List(point).get(0))
    lats = ring.map(lambda point: ee.List(point).get(1))

    return feature.set({
        "min_lon": ee.Number(lons.reduce(ee.Reducer.min())),
        "max_lon": ee.Number(lons.reduce(ee.Reducer.max())),
        "min_lat": ee.Number(lats.reduce(ee.Reducer.min())),
        "max_lat": ee.Number(lats.reduce(ee.Reducer.max()))
    })

basins_bbox = basins.map(add_bbox_metrics)

bbox_fields = [
    "HYBAS_ID", "PFAF_ID", "SUB_AREA",
    "min_lon", "max_lon", "min_lat", "max_lat"
]

bbox_features = (
    basins_bbox
    .select(bbox_fields, None, False)
    .getInfo()["features"]
)

bbox_df = pd.DataFrame([
    f["properties"] for f in bbox_features
])

display(
    bbox_df.sort_values("SUB_AREA")
    .reset_index(drop=True)
    .round(4)
)

,HYBAS_ID,PFAF_ID,SUB_AREA,max_lat,max_lon,min_lat,min_lon
0,4070029760,4540107,87.5,9.6375,76.5083,9.5208,76.3786
1,4070029690,4540103,502.9,9.3125,76.7875,8.9375,76.3792
2,4070029730,4540105,543.9,9.9750,76.4208,9.3167,76.2328
3,4070029780,4540109,660.9,10.1917,76.4795,9.6417,76.1625
4,4070029750,4540106,1318.6,9.8458,76.9208,9.3708,76.3828
5,4070029680,4540102,1507.1,9.1750,77.2625,8.7250,76.5333
6,4071150410,4538029,1543.0,10.7000,77.5083,10.1244,77.0619
7,4070029770,4540108,1809.4,10.1337,76.9417,9.6161,76.3994
8,4070029720,4540104,3955.6,9.6708,77.2833,9.0208,76.3625
9,4070029530,4540101,4728.5,9.0083,77.6500,8.0744,76.5333


## 2.8 Verify the HydroBASINS Dataset Level

Confirm the source collection and its metadata before selecting a catchment.

The current candidates have not yet been matched to the official Pamba River Basin boundary. Dataset-level verification helps avoid selecting polygons from an unintended HydroBASINS layer.

In [12]:
# Inspect the collection source and candidate identifiers.
# This does not change or select any basin.

print("Candidate count:", basins.size().getInfo())

print("\nCandidate HYBAS_ID values:")
print(
    basins.aggregate_array("HYBAS_ID")
    .getInfo()
)

print("\nCandidate PFAF_ID values:")
print(
    basins.aggregate_array("PFAF_ID")
    .getInfo()
)

Candidate count: 14

Candidate HYBAS_ID values:
[4071150410, 4070029250, 4070029360, 4070029440, 4070029680, 4070029720, 4070029750, 4070029770, 4070029810, 4070029530, 4070029690, 4070029730, 4070029760, 4070029780]

Candidate PFAF_ID values:
[4538029, 4539020, 4539060, 4539080, 4540102, 4540104, 4540106, 4540108, 4540200, 4540101, 4540103, 4540105, 4540107, 4540109]


## 2.9 Compare HydroBASINS Levels

Compare Levels 6, 7, and 8 within the existing search area. For each level, inspect the five candidate catchments whose reported sub-basin areas are closest to the published Pamba basin area of approximately 2,235 km².

Area similarity is used only to narrow the candidates. Final selection requires geographic and boundary verification.

In [13]:
REFERENCE_AREA_KM2 = 2235

level_results = []

for level in [6, 7, 8]:
    collection_id = (
        f"WWF/HydroSHEDS/v1/Basins/hybas_{level}"
    )

    candidates = (
        ee.FeatureCollection(collection_id)
        .filterBounds(search_area)
    )

    def add_comparison_metrics(feature):
        centroid = feature.geometry().centroid(
            maxError=100
        ).coordinates()

        area = ee.Number(feature.get("SUB_AREA"))

        return feature.set({
            "level": level,
            "area_difference_km2": area.subtract(
                REFERENCE_AREA_KM2
            ).abs(),
            "centroid_lon": centroid.get(0),
            "centroid_lat": centroid.get(1)
        })

    closest = (
        candidates
        .map(add_comparison_metrics)
        .sort("area_difference_km2")
        .limit(5)
        .select([
            "level",
            "HYBAS_ID",
            "PFAF_ID",
            "SUB_AREA",
            "UP_AREA",
            "area_difference_km2",
            "centroid_lon",
            "centroid_lat"
        ])
    )

    result = closest.getInfo()

    for feature in result["features"]:
        level_results.append(feature["properties"])

    print(
        f"Level {level}: "
        f"{candidates.size().getInfo()} candidates "
        "intersect the search area"
    )

level_df = pd.DataFrame(level_results)

display(
    level_df.sort_values(
        ["level", "area_difference_km2"]
    ).round(3).reset_index(drop=True)
)

Level 6: 6 candidates intersect the search area
Level 7: 14 candidates intersect the search area
Level 8: 45 candidates intersect the search area


,HYBAS_ID,PFAF_ID,SUB_AREA,UP_AREA,area_difference_km2,centroid_lat,centroid_lon,level
0,4060029360,453906,5159.1,5159.1,2924.1,9.397,77.757,6
1,4060029440,453908,5625.5,5625.7,3390.5,8.824,77.588,6
2,4060029810,454020,6376.7,6376.9,4141.7,10.055,76.851,6
3,4060029250,453902,6713.5,6714.3,4478.5,9.890,77.865,6
4,4061148830,453802,9054.8,9054.8,6819.8,10.590,77.628,6
5,4070029770,4540108,1809.4,1809.5,425.6,9.894,76.643,7
6,4071150410,4538029,1543.0,1543.4,692.0,10.389,77.286,7
7,4070029680,4540102,1507.1,1507.1,727.9,8.976,76.946,7
8,4070029750,4540106,1318.6,1318.9,916.4,9.625,76.642,7
9,4070029780,4540109,660.9,660.9,1574.1,9.973,76.344,7


##